In [24]:
import torch 
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer 
from peft import get_peft_model, LoraConfig
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


The CorDA tuning step looked at how to tune a CorDA model efficiently using hugging face's high level tuning library and training loops. In this notebook, I will demonstrate what happens under the hood when tuning a smaller model -- for testing purposes -- and as well as the effectiveness of adapter tuning on downstream tasks. 

### Load the gpt-neo-125M model 

In [23]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125m")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")

### A function to save the model's target params locally

We'll save the params of each block's q, k, v projections 

In [44]:
import torch as pt
import os

EXPERIMENT_NUMBER = 0
proj_modules = ["k_proj", "q_proj", "v_proj"]

def save_model_target_params(target_model: nn.Linear, stage: str = "base_model_attention_proj_weights"):
    """
    Args:
        target_model: the model you want to save params 
        stage: which stage in the experiment we're in. Basically a way to organize model params saved locally 
    """
    GPT_NEO_BLOCK = target_model.transformer.h
    for idx, gpt_block in enumerate(GPT_NEO_BLOCK):
        attention = gpt_block.attn.attention
        block_dir = f"../model/lora/{stage}/block_{idx}"
        os.makedirs(block_dir, exist_ok=True)
        if stage == "base_model_attention_proj_weights":        
            pt.save(attention.k_proj.weight.detach().cpu(), f"{block_dir}/k_proj.pt")
            pt.save(attention.q_proj.weight.detach().cpu(), f"{block_dir}/q_proj.pt")
            pt.save(attention.v_proj.weight.detach().cpu(), f"{block_dir}/v_proj.pt")
        else: 
            pt.save(attention.k_proj.base.weight.detach().cpu(), f"{block_dir}/k_proj.pt")
            pt.save(attention.q_proj.base.weight.detach().cpu(), f"{block_dir}/q_proj.pt")
            pt.save(attention.v_proj.base.weight.detach().cpu(), f"{block_dir}/v_proj.pt")
            


In [22]:
save_model_target_params(target_model=model)

### Define LoRa Initializer

In [28]:
class LoRaHelper(nn.Module):
    def __init__(self, base_layer: nn.Linear, alpha: int, r: int = 4):
        super().__init__()

        self.base = base_layer
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        in_dim = self.base.in_features
        out_dim = self.base.out_features

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Get reference tensor for device/dtype
        ref = self.base.weight
 
        A = torch.empty((r, in_dim), device=ref.device, dtype=ref.dtype)
        A = torch.nn.init.normal_(A, mean=0.0, std=0.02)

        B = torch.zeros((out_dim, r), device=ref.device, dtype=ref.dtype)

        self.A = nn.Parameter(A)
        self.B = nn.Parameter(B)

    def forward(self, x):
     
        result = self.base(x) 
        down_proj = x @ self.A.T
        up_proj = down_proj @ self.B.T

        return result + self.scaling * up_proj


### Define Hyper params 

In [29]:
EPOCHS = 1 
alpha = 0.02
lr = 0.01
batch_size = 2 

### Load the dataset 

In [30]:
from datasets import load_dataset
from utils.dataset import getRawDataset, preprocess
import importlib

# importlib.reload(dataset)
# importlib.reload(text_cleaners)

BATCH_SIZE = 5
datasets_to_test = ["wikitext", "imdb", "sst2", "squad_v2"]

for dataset_name in datasets_to_test:
    print(f"\n=== {dataset_name.upper()} ===")
    
    raw_ds = getRawDataset(dataset_name, split="train")
    
    raw_batch = raw_ds.select(range(BATCH_SIZE))
    
    print("\n--- RAW ---")
    for i, item in enumerate(raw_batch):
        print(f"{i}: {item}")
    
    batch_dict = raw_batch[:]
    # if dataset_name.lower() == 'sst2':
    #     # rename 'sentence' -> 'text' so _preprocessSST2 can work as other datasets
    #     batch_dict = {'text': batch_dict['sentence'], 'label': batch_dict['label']}

    cleaned_batch = preprocess(batch_dict, dataset_name)
    print("\n--- CLEANED ---")
    batch_len = len(next(iter(cleaned_batch.values())))
    for i in range(batch_len):
        cleaned_item = {k: cleaned_batch[k][i] for k in cleaned_batch}
        
        if 'text' in cleaned_item:
            print(f"{i}: {cleaned_item['text']}")
        elif 'sentence' in cleaned_item:
            print(f"{i}: {cleaned_item['sentence']}")
        elif 'context' in cleaned_item:
            print(f"{i}: {cleaned_item['context']}")
        else:
            print(f"{i}: {cleaned_item}")



=== WIKITEXT ===
Loaded wikitext (train): 36718

--- RAW ---
0: {'text': ''}
1: {'text': ' = Valkyria Chronicles III = \n'}
2: {'text': ''}
3: {'text': ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'}
4: {'text': " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it r

### Initialize the LoRa Helper Class for each linear projection head

In [31]:
def add_lora_to_gptneo(model, alpha=16, r=4):
    for block in model.transformer.h:   
        attn = block.attn.attention     
        attn.k_proj = LoRaHelper(attn.k_proj, alpha=alpha, r=r)
        attn.q_proj = LoRaHelper(attn.q_proj, alpha=alpha, r=r)
        attn.v_proj = LoRaHelper(attn.v_proj, alpha=alpha, r=r)
        attn.out_proj = LoRaHelper(attn.out_proj, alpha=alpha, r=r)

    print("LoRA attached to GPT-Neo attention layers.")

In [32]:
add_lora_to_gptneo(model)

LoRA attached to GPT-Neo attention layers.


## Tuning Logic 

In [35]:
from utils import dataset

tokenizer.pad_token = tokenizer.eos_token

# To Optimize training only use the optimizer on trainable params in the network
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad]
    ,lr=2e-5)

loss_fn = torch.nn.CrossEntropyLoss()

datasets= ['wikitext']

#Set model to GPU 
model.to(device)

for dataset_name in datasets:
    ds = dataset.getRawDataset(dataset_name, "train")
    ds = ds.with_format("torch")
    dataloader = DataLoader(ds,batch_size=batch_size, shuffle=True, drop_last=True) 

    for epoch in range(EPOCHS):    
        model.train()

        for step, batch in enumerate(tqdm(dataloader)):      

            cleaned = dataset.preprocess(batch, dataset_name)    

            # Determine which field to tokenize based on dataset type
            if dataset_name.lower() == "squad_v2":
                # SQuAD requires both context and question
                # You can concatenate them with a separator or feed them separately
                tokenized_batch = tokenizer(
                    text=cleaned['question'],
                    text_pair=cleaned['context'],
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                    max_length=384
                ).to(model.device)
            else:
                if len(cleaned['text']) == 0:
                    continue
                # Standard text-based datasets
                tokenized_batch = tokenizer(
                    cleaned['text'],
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                    max_length=96
                ).to(model.device)
            


            input_ids = tokenized_batch["input_ids"].to(model.device)
            attention_mask = tokenized_batch["attention_mask"].to(model.device)
            
            labels = input_ids.clone()
            labels[input_ids == tokenizer.pad_token_id] = -100
            # This is the forward pass
            preds = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels) 
            logits = preds["logits"]
            loss = preds.loss
            
            # # zero out the gradients 
            optimizer.zero_grad()
            # #Run back prop once for the step 
            loss.backward()
            # #Use the defined optimizer to update the grads accordingly 
            # # Tells the optimizer to take a step in gradient descent and update the grads with the provided learning rate
            optimizer.step()
        
        

Loaded wikitext (train): 36718


100%|██████████| 18359/18359 [08:30<00:00, 35.94it/s]


### Save model Params locally in the exp_# directory


In [45]:
save_model_target_params(target_model=model, stage="exp_0")